In [ ]:
import cv2
import json
import pickle
import math
import random
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import Dataset, DataLoader, random_split
from typing import List, Tuple
from tqdm import tqdm
from models import StackedHourglassCBAM
from utils import softargmax_2d
random.seed(41) # 10, 11, 12
#device = torch.device("cuda:0")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
with open("pickle/folds.pkl", "rb") as f:
    folds = pickle.load(f)

In [ ]:
exclude_idx = 1
train_data = sum([fold for i, fold in enumerate(folds) if i != exclude_idx], [])
val_data = folds[exclude_idx]
random.shuffle(train_data)
random.shuffle(val_data)

In [ ]:
def apply_crop(image, center):

    H, W = image.shape

    if H > 5500:
        crop_size = 1024#896#768
        #crop_size = random.randrange(768, 1025, 2)
    else:
        crop_size = 512
        #crop_size = random.randrange(384, 513, 2)
    
    half = crop_size // 2

    x_center, y_center = center

    # Ensure coordinates stay within bounds
    x_center = np.clip(x_center, half, W - half)
    y_center = np.clip(y_center, half, H - half)

    # Bounding box coordinates
    x1 =int(max(0, x_center - half))
    y1 = int(max(0, y_center - half))
    x2 = int(min(W, x_center + half))
    y2 = int(min(H, y_center + half))
    
    cropped_image = image[y1:y2, x1:x2]
    
    return cropped_image, (x1, y1)

In [ ]:
def generate_heatmap(size_hw: Tuple[int, int], center_xy: Tuple[float, float], sigma: float = 2.0):
    """Create a single 2D gaussian heatmap (H,W) with center (x,y) in pixel coords."""
    W, H = size_hw[1], size_hw[0]
    y = torch.arange(H, dtype=torch.float32)
    x = torch.arange(W, dtype=torch.float32)
    yy, xx = torch.meshgrid(y, x, indexing="ij")
    cx, cy = center_xy
    hm = torch.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * sigma ** 2))
    return hm

In [ ]:
LABELS = (
    "R2", "R3", # crops
    "RM1", "RM2",
    "RSL1", "RSM1",
    "LTE", "MTE", # edges
    "LTL", "MTM",
    "LTC", "MTC", # center
    "LTM", "MTL"
)

class CustomDataset(Dataset):
    def __init__(self, data, image_size=(512, 512), heatmap_size=(128, 128), sigma=2, train=False):
        self.data = data
        self.image_size = image_size
        self.heatmap_size = heatmap_size
        self.sigma = sigma
        self.train = train

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        image_path, keypoints = self.data[idx]
        image_array = np.fromfile(image_path, dtype=np.uint8)
        image = cv2.imdecode(image_array, cv2.IMREAD_GRAYSCALE)
        if image is None:
            raise ValueError(f"Failed to read image at {image_path}")
        H, W = image.shape

        if self.train:
            
            if random.random() < 0.5:
                clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
                image = clahe.apply(image)
            # Brightness augmentation
            if random.random() < 0.5:
                factor = random.uniform(0.9, 1.1)  # brightness factor
                image = np.clip(image * factor, 0, 255).astype(np.uint8)

            if random.random() < 0.5:
                if random.random() < 0.5:
                    #print(f"SHARPEN")
                    kernel = np.array([[0, -1, 0],
                                [-1, 5, -1],
                                [0, -1, 0]])
                    
                    sharpened  = cv2.filter2D(image, -1, kernel)
                    alpha = random.uniform(0.1, 0.4)  # blend factor
                    image = cv2.addWeighted(sharpened, alpha, image, 1 - alpha, 0)
            
                else:
                    #print(f"BLUR")
                    ksize = random.choice([3, 5])
                    sigmaX = random.uniform(0.3, 1.0)
                    image = cv2.GaussianBlur(image, (ksize, ksize), sigmaX=sigmaX)
        
        keypoints = [keypoints[l] for l in LABELS]
        keypoints = np.array([(x, (H - y)) for x, y in keypoints], dtype=np.float32)

        crop_center = np.sum(keypoints[:2], axis=0) / 2
        cropped_image, shift = apply_crop(image, crop_center)
        
        resized_image = cv2.resize(cropped_image, (self.image_size[1], self.image_size[0]))
        
        keypoints = keypoints[2:] - shift

        sx = self.heatmap_size[1] / cropped_image.shape[1]
        sy = self.heatmap_size[0] / cropped_image.shape[0]

        keypoints[:, 0] *= sx
        keypoints[:, 1] *= sy

        image_tensor = torch.from_numpy(resized_image).float().unsqueeze(0) / 255.0
        keypoint_tensor = torch.from_numpy(keypoints).float()
        heatmap_tensor = torch.stack([generate_heatmap(self.heatmap_size, kp, sigma = self.sigma) for kp in keypoints])
        meta = {"image_path": image_path}
        
        return image_tensor, keypoint_tensor, heatmap_tensor, meta

In [ ]:
batch_size = 32
image_size = (512, 512)
heatmap_size = (128, 128)

#Shuffle before splitting
#random.shuffle(data)

#train_ratio = 0.8
#total_size = len(data)
#train_size = int(train_ratio * total_size)

train_dataset = CustomDataset(
    data=train_data, # data[:train_size]
    image_size=image_size,
    heatmap_size=heatmap_size,
    sigma=2,
    train=True # True
)

val_dataset = CustomDataset(
    data=val_data, # data[train_size:]
    image_size=image_size,
    heatmap_size=heatmap_size,
    sigma=2,
    train=False
)

# val_dataset = CustomDataset(
#     data=data[train_size:(train_size + int(0.1 * total_size))],
#     image_size=image_size,
#     heatmap_size=heatmap_size,
#     sigma=2,
#     train=False
# )

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
img, gt_kps, gt_hmps, meta = train_dataset[0]  
print(gt_hmps.shape)
img = img.squeeze().numpy()

gt_kps = gt_kps.numpy()
gt_kps[:, 0] *= 512/128
gt_kps[:, 1] *= 512/128

gt_hmps = gt_hmps.numpy()
cgt_hmps = np.max(gt_hmps, axis=0)
rgt_hmps = cv2.resize(cgt_hmps, (512, 512)) # (512, 256)

fig, axes = plt.subplots(1, 3, figsize=(16, 16))

axes[0].imshow(img, cmap='gray')
axes[0].scatter(gt_kps[:, 0], gt_kps[:, 1], c='lime', marker='o', s=30)
axes[0].axis('off')

axes[1].imshow(img, cmap='gray')
axes[1].imshow(rgt_hmps, cmap='jet', alpha=0.5)
axes[1].axis('off')

axes[2].imshow(img, cmap='gray')
axes[2].imshow(rgt_hmps > 0.03, cmap='jet', alpha=0.5)
axes[2].axis('off')

plt.show()

In [ ]:
model = StackedHourglassCBAM(num_keypoints=12, num_stacks=2, depth=4, channels=256, in_ch=1).to(device)
x = torch.rand(1, 1, 512, 512).to(device)
outs = model(x)
print([out.shape for out in outs])
print(f"Model params: {sum(p.numel() for p in model.parameters())}")

In [ ]:
train_losses, val_losses = [], []

num_epochs = 100
warmup_epochs = 3
base_lr = 3e-4 #3e-4
weight_decay = 1e-4
grad_clip_norm  = 1.0
use_amp = True
model_path = "saved/hourglass_cbam_mse[knee]_fold_1.pth"

criterion = nn.MSELoss()
#criterion = nn.BCEWithLogitsLoss()
#criterion = FocalBCELoss(alpha=0.25, gamma=2.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=base_lr, weight_decay=weight_decay)
scaler = GradScaler(enabled=use_amp)

def lr_lambda(current_epoch):
    if current_epoch < warmup_epochs:
        # Linear warmup
        return float(current_epoch + 1) / float(warmup_epochs)
    else:
        # Cosine decay
        progress = (current_epoch - warmup_epochs) / float(num_epochs - warmup_epochs)
        return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = LambdaLR(optimizer, lr_lambda)

best_val_loss = float('inf')
epochs_no_improve = 0
early_stopping_patience = 4

def decode_argmax(hm):
    B, J, H, W = hm.shape
    flat = hm.view(B, J, -1)
    idx = flat.argmax(dim=-1)
    xs = (idx % W).float()
    ys = (idx // W).float()
    return torch.stack([xs, ys], dim=-1)

def pck_like_err(pred_xy, gt_xy):
    return torch.linalg.vector_norm(pred_xy - gt_xy, dim=-1).mean()

for epoch in range(num_epochs):
    model.train()

    train_loss = 0

    for images, _, heatmaps, _ in tqdm(train_loader, desc=f"[Epoch {epoch+1}/{num_epochs}] Training"):
        images = images.to(device, non_blocking=True)
        heatmaps = heatmaps.to(device, non_blocking=True).float()

        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=use_amp):
            outputs = model(images)
            #loss = sum(w * criterion(o, heatmaps) for w, o in zip(head_weights, outputs))
            loss = sum(criterion(o, heatmaps) for o in outputs) / len(outputs)

        if use_amp:
            scaler.scale(loss).backward()
            # Unscale gradients before clipping
            scaler.unscale_(optimizer)
            # Clip the gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
            # Then optimizer step
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
            optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # Validation
    model.eval()
    val_loss = 0
    val_pix_err = 0
    with torch.no_grad():
        for images, keypoints, heatmaps, _ in tqdm(val_loader, desc=f"[Epoch {epoch+1}/{num_epochs}] Validation"):
            images = images.to(device, non_blocking=True)
            gt_xy = keypoints.to(device, non_blocking=True).float()
            heatmaps = heatmaps.to(device, non_blocking=True).float()

            with autocast(enabled=use_amp):
                outputs = model(images)
                #loss = sum(w * criterion(o, heatmaps) for w, o in zip(head_weights, outputs))
                loss = sum(criterion(o, heatmaps) for o in outputs) / len(outputs)
                
                #loss = criterion(outputs, heatmaps)
                pred_xy = decode_argmax(outputs[-1])
                batch_err = pck_like_err(pred_xy, gt_xy)
                val_pix_err += batch_err.item()
                
            val_loss += loss.item()

    val_loss /= len(val_loader)
    val_pix_err /= len(val_loader)
    scheduler.step()
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs} ➤ Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f} | mean_pixel_err: {val_pix_err:.2f}")

    # Early Stopping & Save Best
    if val_loss < best_val_loss:
        print(f"🟢 New best model (val_loss: {val_loss:.6f} < {best_val_loss:.6f}) — saving to {model_path}")
        best_val_loss = val_loss
        torch.save(model.state_dict(), model_path)
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        print(f"🔴 No improvement for {epochs_no_improve} epoch(s)")
    
    if epochs_no_improve >= early_stopping_patience:
        print("⏹ Early stopping triggered.")
        break

In [ ]:
model.eval()

In [ ]:
indices = []

for i in tqdm(range(len(val_dataset))):
    img, gt_kps, gt_hmps, meta = val_dataset[i]

    with torch.no_grad():
        x = img.unsqueeze(0).to(device)
        outs = model(x)

    p_hmps = outs[-1]
    p_kps = softargmax_2d(p_hmps, beta=100.0)  # (1, 12, 2)
    p_kps = p_kps.squeeze().cpu().numpy()
    p_kps[:, 0] *= 512 / 128
    p_kps[:, 1] *= 512 / 128

    p_hmps = p_hmps.squeeze().cpu().numpy()
    p_hmps = np.max(p_hmps, axis=0)
    p_hmps = cv2.resize(p_hmps, (512, 512))

    img = img.squeeze().numpy()
    gt_kps = gt_kps.numpy()
    gt_kps[:, 0] *= 512 / 128
    gt_kps[:, 1] *= 512 / 128

    dist = np.linalg.norm(gt_kps - p_kps, axis=1)

    if any(d > 15 for d in dist):
        indices.append(i)

In [ ]:
print(indices)

In [ ]:
idx = 121
img, gt_kps, gt_hmps, meta = val_dataset[idx]

print(meta['image_path'])

with torch.no_grad():
    x = img.unsqueeze(0).to(device)
    outs = model(x)
    #p_hmps_up = F.interpolate(p_hmps[-1], size=(512, 512), mode='bilinear', align_corners=False)

p_hmps = outs[-1]
p_kps = softargmax_2d(p_hmps, beta=100.0)  # (1, 12, 2)
p_kps = p_kps.squeeze().cpu().numpy()
p_kps[:, 0] *= 512 / 128
p_kps[:, 1] *= 512 / 128


p_hmps = p_hmps.squeeze().cpu().numpy()
p_hmps = np.max(p_hmps, axis=0)
p_hmps = cv2.resize(p_hmps, (512, 512))
# B, K, H, W = p_hmps_up.shape
# kps = []

img = img.squeeze().numpy()
gt_kps = gt_kps.numpy()
gt_kps[:, 0] *= 512 / 128
gt_kps[:, 1] *= 512 / 128

dist = np.linalg.norm(gt_kps - p_kps, axis=1)
print(LABELS[2:])
print(dist)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

axes[0].imshow(img, cmap='gray')
axes[0].scatter(gt_kps[:, 0], gt_kps[:, 1], c='lime', marker='o', s=20)
axes[0].scatter(p_kps[:, 0], p_kps[:, 1], c='red', marker='o', s=20)
 
axes[1].imshow(img, cmap='gray')
axes[1].imshow(p_hmps, cmap='jet', alpha=0.5)

for ax in axes:
    ax.axis('off')
plt.show()

# [2.5204163 1.0026113 2.8162513 8.15204   1.1367732 8.030139  7.6611757
#  3.1140485 5.911936  4.752094  8.985992  7.1919   ]